<div dir="rtl">
<h1>چهار بایت به‌ازای وزن، کل حافظه نیست</h1>
<p>درس 71 از 76 · مدل بزرگ چه هزینه‌های تازه‌ای دارد؟ · <code dir="ltr">63-scale</code></p>
<p><a target="_self" href="http://127.0.0.1:8000/part-10/chapter-01/63-scale.html">📖 بازگشت به همین درس</a></p>
<p>تخمین حافظهٔ وزن را از تخمین سادهٔ آموزش و مصرف واقعی Tensorها جدا کنید.</p><p>پیش‌نیاز: numel، Dtype، Gradient و دو وضعیت Adam.</p>
<p>این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو Cell با برچسب TODO را خودتان کامل کنید. پیام INCOMPLETE یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p>از بالا به پایین اجرا کنید. پس از تغییر هر تابع، Cell آن و سپس Cell آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code>Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl">
<h2>قبل از اجرا، پیش‌بینی کنید</h2>
<p>یک میلیون وزن float32 تقریباً چهار میلیون بایت است؛ چرا آموزش به بیشتر از همین مقدار نیاز دارد؟</p>
</div>

<div dir="rtl"><p>پیش‌بینی من: …</p></div>

In [ ]:
import torch
from mini_gpt.config import ModelConfig
from mini_gpt.model import MiniGPT
torch.set_num_threads(1)
torch.manual_seed(7)
model = MiniGPT(ModelConfig(12,4,8,2,1,0.0))
optimizer = torch.optim.AdamW(model.parameters())
model(torch.tensor([[1,2,3]]),torch.tensor([[2,3,4]]))[1].backward()
optimizer.step()
parameter_count = sum(p.numel() for p in model.parameters())
optimizer_bytes = sum(value.numel()*value.element_size() for state in optimizer.state.values()
                      for value in state.values() if isinstance(value,torch.Tensor))
print('parameters:',parameter_count,'actual optimizer tensor bytes:',optimizer_bytes)
print('Activation memory and allocator overhead are not measured here.')

<div dir="rtl">
<h2>این بار شما کد بنویسید</h2>
<p>memory_estimate(parameters, bytes_per_number, Training) یک تعداد بایت صحیح بدهد. برای وزن‌ها یک مجموعه و برای تخمین سادهٔ آموزش چهار مجموعهٔ هم‌اندازه در نظر بگیرید: وزن، Gradient و دو وضعیت Adam. این مدل تخمینی فرض می‌کند Dtype هر چهار یکسان است.</p>
</div>

In [ ]:
def memory_estimate(parameters, bytes_per_number, training):
    # TODO: تخمین خام، نه حافظهٔ اوج واقعی
    return None

In [ ]:
def test_exercise():
    result = memory_estimate(1_000_000,4,False)
    if result is None:
        return False
    assert result==4_000_000
    assert memory_estimate(1_000_000,4,True)==16_000_000
    assert memory_estimate(100_000_000,4,False)==400_000_000
    assert memory_estimate(10,2,True)==80
    assert memory_estimate(0,4,True)==0
    return True
exercise_complete = test_exercise()
print('PASS' if exercise_complete else 'INCOMPLETE: memory_estimate')

<div dir="rtl">
<h2>فقط یک عامل را تغییر دهید</h2>
<p>فقط تعداد بایت هر وزن را در تخمین تغییر دهید. این محاسبه مدل را Quantize نمی‌کند و سرعت اجرا را هم اندازه نمی‌گیرد.</p>
</div>

In [ ]:
for bytes_per_weight in (4,2,1):
    print(bytes_per_weight,parameter_count*bytes_per_weight,'estimated raw weight bytes')

<div dir="rtl">
<h2>خرابی را پیدا کنید</h2>
<p>len(model.parameters()) تعداد ظرف‌هاست، نه تعداد وزن‌ها. model_weight_bytes(Model) مجموع numel*element_size همهٔ Parameterها را بدهد؛ Bufferها و Activationها جزو این تابع نیستند.</p>
</div>

In [ ]:
wrong = len(list(model.parameters()))*4
print('wrong bytes from parameter-object count:',wrong)
print('first weight shape:',tuple(next(model.parameters()).shape))

<div dir="rtl">
<h2>اصلاح را خودتان بنویسید</h2>
<p>علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def model_weight_bytes(model):
    # TODO: اندازهٔ هر Tensor و dtype واقعی آن
    return None

In [ ]:
def test_repair():
    result = model_weight_bytes(model)
    if result is None:
        return False
    assert result==parameter_count*4
    layer = torch.nn.Linear(3,2,bias=True).double()
    assert model_weight_bytes(layer)==(3*2+2)*8
    assert model_weight_bytes(torch.nn.Identity())==0
    return True
repair_complete = test_repair()
print('PASS' if repair_complete else 'INCOMPLETE: model_weight_bytes')

<div dir="rtl">
<h2>در Mini-GPT کجا به کار می‌آید؟</h2>
<p>وزن‌ها و وضعیت Optimizer از مدل واقعی‌اند؛ برآورد چهارمجموعه‌ای فقط یک مدل ساده است. Mixed precision، آموزش توزیع‌شده و FlashAttention در این دفتر اجرا نمی‌شوند.</p>
</div>

<div dir="rtl">
<h2>با زبان خودتان توضیح دهید</h2>
<p>کدام هزینه‌ها هنوز در تخمین شما نیستند و چرا کم‌شدن بایت هر وزن الزاماً سرعت را بیشتر نمی‌کند؟</p>
</div>
<div dir="rtl"><p>پیش‌بینی و مشاهدهٔ من: …</p><p>علت خرابی و اصلاح من: …</p></div>

<div dir="rtl"><p><a target="_self" href="http://127.0.0.1:8000/part-10/chapter-01/63-scale.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/63-scale.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>